# ALL Met Painting Eyes

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!mkdir json && wget -P json https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/json/mp_masks_definitions.json
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data
!cd data && git config user.name "Thiago Hersan" && git config user.email "thiago.hersan+github@gmail.com"

In [ ]:
from utils import get_combined_jsons

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"

JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

## Painting Objects

- $15\text{,}178$ on website
- $15\text{,}050$ (±100) available in API
- $14\text{,}233$ (±50) have images (according to `hasImages=true` API query param)
- $9\text{,}015$ ($63\%$) actually have images that can be downloaded
- $5\text{,}000$ ($55\%$) have faces
- $4\text{,}498$ ($90\%$) have extractable eyes
- $3\text{,}640$ ($80\%$) are from color images

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

## Get Object Metadata

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

for cnt,oid in enumerate(obj_ids):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  mPU.get_obj_data(oid)

## Get Faces, Landmarks and Eyes

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR, with_detectors=True)
objects_data = get_combined_jsons(mPU.json_objs_dir)
len(objects_data)

In [ ]:
for cnt,obj_data in enumerate(objects_data):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(objects_data)}")

  if mPU.is_processed(obj_data):
    continue

  img = mPU.get_image(obj_data["primaryImage"])
  if img is None:
    continue

  face_data = mPU.get_face_data(obj_data, img)
  if face_data is None:
    continue

  landmark_data = mPU.get_landmark_data(face_data, img)
  if landmark_data is None:
    continue

  mPU.get_eye_images(landmark_data, img)

## Checks

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
objs = get_combined_jsons(mPU.json_objs_dir)
faces = get_combined_jsons(mPU.json_faces_dir)
landmarks = get_combined_jsons(mPU.json_landmarks_dir)

no_imgs = set(mPU.no_imgs)
ye_imgs = set([o["objectID"] for o in objs])

no_faces = set(mPU.no_faces)
ye_faces = set([o["objectID"] for o in faces])

no_lands = set(mPU.no_landmarks)
ye_lands = set([o["objectID"] for o in landmarks])

print(len(no_imgs), len(ye_imgs), len(no_imgs) + len(ye_imgs))
print(len(no_faces), len(ye_faces), len(no_faces) + len(ye_faces))
print(len(no_lands), len(ye_lands), len(no_lands) + len(ye_lands))

print(len(no_imgs.intersection(ye_imgs)))
print(len(no_faces.intersection(ye_faces)))
print(len(no_lands.intersection(ye_lands)))

## Export csv

In [ ]:
from utils_paintings import PaintingsUtils

DATA_DIR = "./data"

CSV_DIR = f"{DATA_DIR}/csv"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

mPU.export_csv(mPU.json_objs_dir, f"{CSV_DIR}/objects.csv")
mPU.export_csv(mPU.json_faces_dir, f"{CSV_DIR}/faces.csv")
mPU.export_csv(mPU.json_landmarks_dir, f"{CSV_DIR}/landmarks.csv")
mPU.export_csv(f"{JSON_DIR}/landmarks-color", f"{CSV_DIR}/landmarks-color.csv")

## Rotate/Align

In [ ]:
import json
import numpy as np

from os import listdir, makedirs, path
from PIL import Image as PImage

from utils import export_combined_jsons

DATA_DIR = "./data"

CSV_DIR = f"{DATA_DIR}/csv"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

JSON_COLOR_DIR = f"{JSON_DIR}/landmarks-color"
IMG_COLOR_DIR = f"{IMG_DIR}/eyes-color"
IMG_ALIGNED_DIR = f"{IMG_DIR}/eyes-color-aligned-01"

makedirs(IMG_ALIGNED_DIR, exist_ok=True)

In [ ]:
PUPIL_L_IDX, PUPIL_R_IDX = 473, 468

oids = sorted([f.split(".")[0] for f in listdir(JSON_COLOR_DIR) if f.endswith("json")])

for icnt,oid in enumerate(oids):
  if icnt % 64 == 0:
    print(icnt, "/", len(oids))

  with open(f"{JSON_COLOR_DIR}/{oid}.json", "r") as ifp:
    obj_data = json.load(ifp)

  for fcnt,landmarks in enumerate(obj_data["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    face_cnt_str = f"000{fcnt}"[-3:]
    img_in_path = f"{IMG_COLOR_DIR}/{oid}_{face_cnt_str}.avif"
    img_out_path = f"{IMG_ALIGNED_DIR}/{oid}_{face_cnt_str}.avif"

    if path.isfile(img_out_path):
      continue

    landmarks_np = np.array(landmarks) * [obj_data["img_ratio"], 1.0]
    R2L = landmarks_np[PUPIL_L_IDX] - landmarks_np[PUPIL_R_IDX]
    angle_deg = np.degrees(np.arctan2(R2L[1], R2L[0]))

    img = PImage.open(img_in_path)
    if abs(angle_deg) > 0.5:
      img.rotate(angle_deg, resample=PImage.Resampling.BICUBIC, expand=True).save(img_out_path)
    else:
      img.save(img_out_path)